# RAG

In [ ]:
import time
start_time = time.time()

### Upload html documents from local folder

BSHTMLLoader: Strips all HTML immediately → tables become unformatted text  
Your custom function: Converts tables to markdown first → tables remain structured

In [ ]:
# Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id + 65456 
        }
    )
    documents.append(doc)

type(documents)     -> list
type(documents[0])  -> # Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id + 65456 
        }
    )
    documents.append(doc)

# type(documents)     -> list
# type(documents[0])  -> langchain_core.documents.base.Document
# documents[0].metadata ->  # Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id + 65456 
        }
    )
    documents.append(doc)

type(documents)     -> list
type(documents[0])  -> # Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id + 65456 
        }
    )
    documents.append(doc)

# type(documents)     -> list
# type(documents[0])  -> langchain_core.documents.base.Document
# documents[0].metadata ->  {'source': '6k_filings\\CIK0000932782_0000932782-23-000006_pemex_fsx6kxq1-2023.htm',    'docid': 65456}

### Split LangChain document objects into chunks that are as well LangChain document objects

Considered using MarkdownHeaderTextSplitter because used markdowns to clarify tables. Still, better RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,    #  adds where each chunk starts in the original document
    separators=["\n\n", "\n", ". ", " ", ""]  # Splits on paragraphs, then lines, then sentences, then words, then characters
)
chunks = text_splitter.split_documents(documents)
print(f'Split {len(documents)} filings (documents) into {len(chunks)} chunks.' )

# type(chunks)     -> list
# type(chunks[0])  -> langchain_core.documents.base.Document
# chunks[0].metadata    ->      {'source': '6k_filings\\CIK0000932782_0000932782-23-000006_pemex_fsx6kxq1-2023.htm',    'docid': 65456,    'start_index': 0}

Split 1010 filings (documents) into 63427 chunks.


### Batch Embeddings
#### Prepare batches

In [7]:
import json
from pathlib import Path

def create_batch_jsonl(
    chunks, 
    output_dir="batch_files",
    max_lines_per_file=10000
):
    """
    Create JSONL files for OpenAI batch embeddings with custom IDs and file limits
    
    Args:
        chunks: List of LangChain Document objects
        output_dir: Directory to save batch files
        max_lines_per_file: Maximum number of tasks per JSONL file
    
    Returns:
        List of created file paths
    """
    Path(output_dir).mkdir(exist_ok=True)
    batch_files = []
    
    # Split chunks into batches
    for batch_num in range(0, len(chunks), max_lines_per_file):
        batch_chunks = chunks[batch_num:batch_num + max_lines_per_file]
        output_file = f"{output_dir}/batch_for_embeddings_{batch_num // max_lines_per_file + 1}.jsonl"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for chunk in batch_chunks:
                # Create unique custom_id from metadata
                custom_id = (
                    str(chunk.metadata['docid']) + "_" + 
                    str(chunk.metadata['start_index'])
                )
                
                out_dict = {
                    "custom_id": custom_id,
                    "method": "POST",
                    "url": "/v1/embeddings",
                    "body": {
                        "model": "text-embedding-3-small",
                        "input": chunk.page_content
                    }
                }
                f.write(json.dumps(out_dict, ensure_ascii=False) + '\n')
        
        batch_files.append(output_file)
        print(f"Created {output_file} with {len(batch_chunks)} tasks")
    
    print(f"\nTotal: {len(batch_files)} batch file(s) created")
    return batch_files

# Create batch files
batch_files = create_batch_jsonl(chunks, max_lines_per_file=10000)

Created batch_files/batch_for_embeddings_1.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_2.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_3.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_4.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_5.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_6.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_7.jsonl with 3427 tasks

Total: 7 batch file(s) created


### Upload input file

In [8]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [9]:
from openai import OpenAI

client = OpenAI()
files = client.files.list()

In [10]:
len(files.to_dict()['data'])

381

In [17]:
from glob import glob

batch_files = glob('./batch_files/batch_for_embeddings_*.jsonl')
batch_files

['./batch_files\\batch_for_embeddings_5.jsonl',
 './batch_files\\batch_for_embeddings_6.jsonl',
 './batch_files\\batch_for_embeddings_7.jsonl']

In [18]:
from tqdm import tqdm
client = OpenAI()

my_batch_files_ids = []
for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    my_batch_files_ids.append(batch_input_file.id)
    print(batch_input_file)

 33%|███▎      | 1/3 [00:07<00:15,  7.68s/it]

FileObject(id='file-LwEmLnDoGyCgg3m7bJyce4', bytes=9082916, created_at=1762883662, filename='batch_for_embeddings_5.jsonl', object='file', purpose='batch', status='processed', expires_at=1765475662, status_details=None)


 67%|██████▋   | 2/3 [00:14<00:07,  7.30s/it]

FileObject(id='file-5dBSggEZuv6jT9ug7So4ET', bytes=9371172, created_at=1762883669, filename='batch_for_embeddings_6.jsonl', object='file', purpose='batch', status='processed', expires_at=1765475669, status_details=None)


100%|██████████| 3/3 [00:18<00:00,  6.12s/it]

FileObject(id='file-Q2SWd38QYnpVor9aMPcPYX', bytes=3394648, created_at=1762883672, filename='batch_for_embeddings_7.jsonl', object='file', purpose='batch', status='processed', expires_at=1765475672, status_details=None)


In [ ]:
my_batch_files_ids

# ['file-3p8GPo2juGoNL8xzdSyUa7',
#  'file-9w1k1gJd78gEqtQdgyCWr4',
#  'file-Rh9eZcVeFz3SxWbcELmuUA',
#  'file-PH3HHc7Ly1q9MQD1UPnsY1']

# ['file-LwEmLnDoGyCgg3m7bJyce4',
#  'file-5dBSggEZuv6jT9ug7So4ET',
#  'file-Q2SWd38QYnpVor9aMPcPYX']

['file-LwEmLnDoGyCgg3m7bJyce4',
 'file-5dBSggEZuv6jT9ug7So4ET',
 'file-Q2SWd38QYnpVor9aMPcPYX']

In [14]:
my_id = 'antonio_m_lancuentra'

In [20]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
batch_description = f"Content embeddings ({my_id}) {timestamp}"

for file_id in tqdm(my_batch_files_ids):
    client.batches.create(
            input_file_id = file_id,
            endpoint="/v1/embeddings",
            completion_window="24h",
            metadata={
                "description": batch_description,
                "timestamp": timestamp
            }
        )

100%|██████████| 3/3 [00:01<00:00,  2.20it/s]


In [ ]:
batch_description
# 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:52:16'
# 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11'

'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11'

### After launching batches, I turn the computer off. Then I need to run the cells below

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
client.files.list().to_dict()

In [27]:
client.batches.list().to_dict()

{'data': [{'id': 'batch_69137880f3b08190a9ee338c8119a903',
   'completion_window': '24h',
   'created_at': 1762883712,
   'endpoint': '/v1/embeddings',
   'input_file_id': 'file-Q2SWd38QYnpVor9aMPcPYX',
   'object': 'batch',
   'status': 'failed',
   'cancelled_at': None,
   'cancelling_at': None,
   'completed_at': None,
   'error_file_id': None,
   'errors': {'data': [{'code': 'duplicate_custom_id',
      'line': 690,
      'message': 'The custom_id for this request is a duplicate of another request. The custom_id parameter must be unique for each request in a batch.',
      'param': 'custom_id'}],
    'object': 'list'},
   'expired_at': None,
   'expires_at': 1762970112,
   'failed_at': 1762883714,
   'finalizing_at': None,
   'in_progress_at': None,
   'metadata': {'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11',
    'timestamp': '2025-11-11 12:55:11'},
   'model': None,
   'output_file_id': None,
   'request_counts': {'completed': 0, 'failed': 0, 'to

In [25]:
batch_description = 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11'

In [26]:
batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id']}  
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_info

[{'batch_id': 'batch_69137880f3b08190a9ee338c8119a903',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11',
  'status': 'failed',
  'request_counts': {'completed': 0, 'failed': 0, 'total': 0},
  'output_file_id': None},
 {'batch_id': 'batch_69137880952481908c4f57271acbaaf4',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11',
  'status': 'failed',
  'request_counts': {'completed': 0, 'failed': 0, 'total': 0},
  'output_file_id': None},
 {'batch_id': 'batch_691378805ec88190972891ecbebf5303',
  'description': 'Content embeddings (antonio_m_lancuentra) 2025-11-11 12:55:11',
  'status': 'failed',
  'request_counts': {'completed': 0, 'failed': 0, 'total': 0},
  'output_file_id': None}]

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.2f} seconds")